In [9]:
import sys
import os

print("=== 5CLOUD — VELIB SILVER & GOLD ===")
print("Python :", sys.version.split()[0])
print("Environnement :", sys.executable)
print("Notebook initialisé avec succès !")

=== 5CLOUD — VELIB SILVER & GOLD ===
Python : 3.11.0
Environnement : c:\Users\cyrin\OneDrive\Desktop\supinfo M2\5CLOUD\azure-realtime-data-platform\.venv\Scripts\python.exe
Notebook initialisé avec succès !


In [10]:

import os
import sys

# Configuration Spark pour Windows
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("5CLOUD-Velib-Silver-Gold")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark initialisé avec succès !")
print("Version Spark :", spark.version)
print("Mode :", spark.sparkContext.master)


Spark initialisé avec succès !
Version Spark : 3.5.5
Mode : local[2]


In [11]:

import json
from pathlib import Path

# Retrouver la racine du projet
project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

BATCH_ID = "0610bc1b-5d65-462b-8a84-335af2737c68"

bronze_path = (
    project_root
    / "data"
    / "bronze"
    / "velib"
    / f"batch_id={BATCH_ID}"
)

pages_path = bronze_path / "pages"

# Trouver les fichiers JSON
json_files = sorted(pages_path.glob("page_*.json"))

print("Pages JSON trouvées :", len(json_files))

# Charger toutes les stations depuis la clé "results"
stations = []

for file_path in json_files:
    with file_path.open(
        "r",
        encoding="utf-8-sig"
    ) as file:
        page_data = json.load(file)

    results = page_data["results"]

    if not isinstance(results, list):
        raise ValueError(
            f"Format results invalide : {file_path.name}"
        )

    stations.extend(results)

print("Stations chargées :", len(stations))

# Vérifications
assert len(json_files) == 16, "Nombre de pages incorrect"
assert len(stations) == 1520, "Nombre de stations incorrect"

print("Chargement Bronze réussi !")



Pages JSON trouvées : 16
Stations chargées : 1520
Chargement Bronze réussi !


In [12]:

import json

# Convertir les stations en JSON
json_records = [
    json.dumps(station, ensure_ascii=False)
    for station in stations
]

# Créer le DataFrame Spark
stations_df = spark.read.json(
    spark.sparkContext.parallelize(json_records, 2)
)

print("=== BRONZE : DATAFRAME SPARK ===")

print("Nombre de stations :", stations_df.count())
print("Nombre de colonnes :", len(stations_df.columns))

# Afficher les premières stations
stations_df.select(
    "stationcode",
    "name",
    "capacity",
    "numbikesavailable",
    "numdocksavailable"
).show(5, truncate=False)

assert stations_df.count() == 1520

print("DataFrame Spark créé avec succès !")


=== BRONZE : DATAFRAME SPARK ===
Nombre de stations : 1520
Nombre de colonnes : 15
+-----------+-------------------------------+--------+-----------------+-----------------+
|stationcode|name                           |capacity|numbikesavailable|numdocksavailable|
+-----------+-------------------------------+--------+-----------------+-----------------+
|1001       |Quai de l'Horloge - Pont Neuf  |17      |14               |0                |
|1002       |Victoria - Place du Chatelet   |22      |8                |10               |
|1003       |Saint-Denis - Rivoli           |43      |29               |17               |
|1006       |Grande Truanderie - Saint-Denis|33      |24               |8                |
|1007       |Turbigo - Française            |16      |2                |14               |
+-----------+-------------------------------+--------+-----------------+-----------------+
only showing top 5 rows

DataFrame Spark créé avec succès !


In [13]:
import sys
import json
from datetime import datetime
from pathlib import Path

# Permettre l'import des modules du projet
project_path = str(project_root.resolve())

if project_path not in sys.path:
    sys.path.insert(0, project_path)

from src.platform_project.cleaning import clean_velib_data

# Charger l'heure de référence du manifeste Bronze
manifest_path = bronze_path / "manifest.json"

with manifest_path.open("r", encoding="utf-8-sig") as file:
    manifest = json.load(file)

collection_time = datetime.fromisoformat(
    manifest["manifest_written_at_utc"].replace("Z", "+00:00")
)

# Appliquer les transformations Silver
silver_df = clean_velib_data(
    stations_df,
    BATCH_ID,
    collection_time
)

print("=== SILVER : TRANSFORMATION ===")
print("Stations Silver :", silver_df.count())

silver_df.select(
    "station_id",
    "station_name",
    "bikes_available",
    "docks_available",
    "capacity_mismatch",
    "is_stale"
).show(5, truncate=False)

assert silver_df.count() == 1520

print("Transformation Silver réussie !")


=== SILVER : TRANSFORMATION ===
Stations Silver : 1520
+----------+-------------------------------+---------------+---------------+-----------------+--------+
|station_id|station_name                   |bikes_available|docks_available|capacity_mismatch|is_stale|
+----------+-------------------------------+---------------+---------------+-----------------+--------+
|1001      |Quai de l'Horloge - Pont Neuf  |14             |0              |true             |false   |
|1002      |Victoria - Place du Chatelet   |8              |10             |true             |false   |
|1003      |Saint-Denis - Rivoli           |29             |17             |true             |false   |
|1006      |Grande Truanderie - Saint-Denis|24             |8              |true             |false   |
|1007      |Turbigo - Française            |2              |14             |false            |false   |
+----------+-------------------------------+---------------+---------------+-----------------+--------+
only show

In [14]:
from pyspark.sql import functions as F
from src.platform_project.quality import check_data_quality

# Appliquer les règles de qualité
valid_df, rejected_df = check_data_quality(silver_df)

print("=== SILVER : VALIDATION QUALITÉ ===")

total_stations = silver_df.count()
valid_count = valid_df.count()
rejected_count = rejected_df.count()

print("Stations analysées :", total_stations)
print("Stations valides :", valid_count)
print("Stations rejetées :", rejected_count)

# Vérifier qu'aucune station n'est perdue
assert valid_count + rejected_count == total_stations

# Afficher les motifs des rejets
print("\nMotifs de rejet :")

rejected_df.select(
    "station_id",
    "quality_errors"
).show(10, truncate=False)

print("Validation qualité Silver réussie !")


=== SILVER : VALIDATION QUALITÉ ===
Stations analysées : 1520
Stations valides : 1520
Stations rejetées : 0

Motifs de rejet :
+----------+--------------+
|station_id|quality_errors|
+----------+--------------+
+----------+--------------+

Validation qualité Silver réussie !


In [15]:
from src.platform_project.silver_history import prepare_silver_history

print("=== SILVER : DÉDUPLICATION ===")

# Dédupliquer les stations valides du lot
silver_history_df = prepare_silver_history(valid_df)

before_count = valid_df.count()
after_count = silver_history_df.count()

print("Stations avant déduplication :", before_count)
print("Stations après déduplication :", after_count)
print("Doublons supprimés :", before_count - after_count)

# Vérifier l'unicité des stations dans le lot
unique_stations = silver_history_df.select(
    "batch_id", "station_id"
).distinct().count()

assert after_count == unique_stations
assert after_count == 1520

print("Déduplication Silver réussie !")


=== SILVER : DÉDUPLICATION ===
Stations avant déduplication : 1520
Stations après déduplication : 1520
Doublons supprimés : 0
Déduplication Silver réussie !


In [16]:
from pyspark.sql import functions as F

from src.platform_project.aggregations import (
    build_gold_global,
    build_gold_communes
)

print("=== GOLD : KPI GLOBAUX ===")

# Utiliser les stations Silver validées et dédupliquées
gold_global_df = build_gold_global(silver_history_df)

gold_global_df.show(truncate=False)

print("\n=== GOLD : KPI PAR COMMUNE ===")

gold_communes_df = build_gold_communes(silver_history_df)

gold_communes_df.orderBy(
    F.desc("total_bikes")
).show(10, truncate=False)

# Vérifications
global_kpi = gold_global_df.first()

assert global_kpi["total_stations"] == 1520
assert global_kpi["total_bikes"] == 18792
assert global_kpi["stale_stations"] == 17
assert global_kpi["capacity_mismatches"] == 848

communes_count = gold_communes_df.count()
print("\nNombre de communes :", communes_count)

assert communes_count == 69

print("Calcul des KPI Gold réussi !")


=== GOLD : KPI GLOBAUX ===
+------------------------------------+--------------+-----------+----------------+--------------+---------------+--------------+-------------+--------------+-------------------+--------------------+
|batch_id                            |total_stations|total_bikes|mechanical_bikes|electric_bikes|available_docks|empty_stations|full_stations|stale_stations|capacity_mismatches|latest_source_update|
+------------------------------------+--------------+-----------+----------------+--------------+---------------+--------------+-------------+--------------+-------------------+--------------------+
|0610bc1b-5d65-462b-8a84-335af2737c68|1520          |18792      |11468           |7324          |29584          |42            |26           |17            |848                |2026-10-28 21:25:31 |
+------------------------------------+--------------+-----------+----------------+--------------+---------------+--------------+-------------+--------------+-------------------+

In [18]:
from src.platform_project.silver_storage import save_silver_local
from src.platform_project.gold_storage import save_gold_local

print("=== SAUVEGARDE SILVER & GOLD ===")

# Enregistrer les données Silver
silver_file = save_silver_local(
    silver_history_df,
    BATCH_ID,
    output_root=str(project_root / "data/silver/velib")
)

# Enregistrer les KPI Gold globaux
gold_global_file = save_gold_local(
    gold_global_df,
    BATCH_ID,
    "global",
    output_root=str(project_root / "data/gold/velib")
)

# Enregistrer les KPI Gold par commune
gold_communes_file = save_gold_local(
    gold_communes_df,
    BATCH_ID,
    "communes",
    output_root=str(project_root / "data/gold/velib")
)

# Vérifier que les fichiers existent
assert silver_file.exists()
assert gold_global_file.exists()
assert gold_communes_file.exists()

print("\nLes 3 fichiers ont été enregistrés !")
print("Silver :", silver_file.name)
print("Gold Global :", gold_global_file.name)
print("Gold Communes :", gold_communes_file.name)
print("Sauvegarde locale réussie !")


=== SAUVEGARDE SILVER & GOLD ===
Silver sauvegardé : c:\Users\cyrin\OneDrive\Desktop\supinfo M2\5CLOUD\azure-realtime-data-platform\data\silver\velib\batch_id=0610bc1b-5d65-462b-8a84-335af2737c68\stations.jsonl
Observations enregistrées : 1520
Gold sauvegardé : c:\Users\cyrin\OneDrive\Desktop\supinfo M2\5CLOUD\azure-realtime-data-platform\data\gold\velib\global\batch_id=0610bc1b-5d65-462b-8a84-335af2737c68\data.jsonl
Gold sauvegardé : c:\Users\cyrin\OneDrive\Desktop\supinfo M2\5CLOUD\azure-realtime-data-platform\data\gold\velib\communes\batch_id=0610bc1b-5d65-462b-8a84-335af2737c68\data.jsonl

Les 3 fichiers ont été enregistrés !
Silver : stations.jsonl
Gold Global : data.jsonl
Gold Communes : data.jsonl
Sauvegarde locale réussie !


In [19]:
print("=== CONFIGURATION DES CHEMINS AZURE ===")

# Stockage ADLS Gen2 créé pour le projet
storage_account = "stvelibchifa2026"
container = "velib-data"

# Racine du stockage Azure
adls_root = (
    f"abfss://{container}@{storage_account}.dfs.core.windows.net"
)

# Chemins Bronze / Silver / Gold
azure_bronze_path = (
    f"{adls_root}/bronze/velib/batch_id={BATCH_ID}"
)

azure_silver_path = (
    f"{adls_root}/silver/velib/batch_id={BATCH_ID}"
)

azure_gold_global_path = (
    f"{adls_root}/gold/velib/global/batch_id={BATCH_ID}"
)

azure_gold_communes_path = (
    f"{adls_root}/gold/velib/communes/batch_id={BATCH_ID}"
)

print("Bronze :", azure_bronze_path)
print("Silver :", azure_silver_path)
print("Gold Global :", azure_gold_global_path)
print("Gold Communes :", azure_gold_communes_path)

print("\nChemins Azure préparés avec succès !")


=== CONFIGURATION DES CHEMINS AZURE ===
Bronze : abfss://velib-data@stvelibchifa2026.dfs.core.windows.net/bronze/velib/batch_id=0610bc1b-5d65-462b-8a84-335af2737c68
Silver : abfss://velib-data@stvelibchifa2026.dfs.core.windows.net/silver/velib/batch_id=0610bc1b-5d65-462b-8a84-335af2737c68
Gold Global : abfss://velib-data@stvelibchifa2026.dfs.core.windows.net/gold/velib/global/batch_id=0610bc1b-5d65-462b-8a84-335af2737c68
Gold Communes : abfss://velib-data@stvelibchifa2026.dfs.core.windows.net/gold/velib/communes/batch_id=0610bc1b-5d65-462b-8a84-335af2737c68

Chemins Azure préparés avec succès !


In [20]:
from src.platform_project.azure_storage import (
    write_parquet_to_adls,
    read_parquet_from_adls
)

print("=== MODULE AZURE STORAGE ===")
print("Fonction écriture :", write_parquet_to_adls.__name__)
print("Fonction lecture :", read_parquet_from_adls.__name__)
print("Module Azure Storage importé avec succès !")

=== MODULE AZURE STORAGE ===
Fonction écriture : write_parquet_to_adls
Fonction lecture : read_parquet_from_adls
Module Azure Storage importé avec succès !


In [21]:
print("=== CONFIGURATION ENVIRONNEMENT ===")

# Mode actuel : développement local
EXECUTION_MODE = "local"

# Les traitements Azure seront activés après
# la configuration du workspace Synapse.
if EXECUTION_MODE == "local":
    print("Mode : développement local")
    print("Stockage : fichiers locaux")
    print("Connexion Azure : non activée")

elif EXECUTION_MODE == "synapse":
    print("Mode : Azure Synapse")
    print("Stockage : ADLS Gen2")
    print("Chemins Azure prêts")

else:
    raise ValueError("Mode d'exécution inconnu")

print("Configuration de l'environnement réussie !")


=== CONFIGURATION ENVIRONNEMENT ===
Mode : développement local
Stockage : fichiers locaux
Connexion Azure : non activée
Configuration de l'environnement réussie !
